In [1]:
import time
import numpy as np

In [ ]:
from tdmclient import ClientAsync
client = ClientAsync()
node = await client.wait_for_node()
await node.lock()

In [ ]:
from threading import Timer

class RepeatedTimer(object):
    def __init__(self, interval, function, *args, **kwargs):
        self._timer     = None
        self.interval   = interval
        self.function   = function
        self.args       = args
        self.kwargs     = kwargs
        self.is_running = False
        self.start()

    def _run(self):
        self.is_running = False
        self.start()
        self.function(*self.args, **self.kwargs)

    def start(self):
        if not self.is_running:
            self._timer = Timer(self.interval, self._run)
            self._timer.start()
            self.is_running = True

    def stop(self):
        self._timer.cancel()
        self.is_running = False

In [4]:
n = 500

In [ ]:
Q = []
R = []

In [17]:
x = [-2.1, -1,  4.3]
y = [3,  1.1,  0.12]
X = np.stack((x, y), axis=0)
X
np.cov(X), np.cov(x, y)
# np.cov(x)

(array([[11.71      , -4.286     ],
        [-4.286     ,  2.14413333]]),
 array([[11.71      , -4.286     ],
        [-4.286     ,  2.14413333]]))

In [34]:
x = np.array([13+3/70, 13+2/57, 13-2/43, 13-3/28, 13+1/15])
y = np.array([27+3/60, 13+2/67, 13-2/33, 13-3/38, 13+1/25])
theta = np.array([np.pi+1/24, np.pi-1/18, np.pi-1/13, np.pi+1/14, np.pi+1/11])
theta = np.array([-np.pi+1/24, np.pi-1/18, np.pi-1/13, -np.pi+1/14, -np.pi+1/11])

In [35]:
dth = np.angle(np.exp(1j*(theta - np.angle(np.mean(np.exp(1j*theta))))))
dth, np.cov(dth), np.cov(theta)

(array([ 0.02734671, -0.06987551, -0.09124304,  0.05710861,  0.07658913]),
 array(0.00577075),
 array(11.34322029))

In [38]:
np.round(np.cov(np.stack((x, y, theta), axis=0)), 3)
# np.round(np.cov(np.stack((x, y, dth), axis=0)), 3)

array([[ 5.0000e-03,  1.6100e-01, -1.2000e-02],
       [ 1.6100e-01,  3.9581e+01, -8.7360e+00],
       [-1.2000e-02, -8.7360e+00,  1.1343e+01]])

In [33]:
np.round(np.cov(np.stack((x-x.mean(), y-y.mean(), dth), axis=0)), 3)

array([[5.0000e-03, 1.6100e-01, 0.0000e+00],
       [1.6100e-01, 3.9581e+01, 9.7000e-02],
       [0.0000e+00, 9.7000e-02, 6.0000e-03]])

In [21]:
np.angle(np.mean(np.exp(1j*theta)))

-3.127272694512819

In [ ]:
def kalman_pos(pos_est_old, pos_msmts, speed, P_prev, dt, Q=Q, R=R):
    
    return 

In [7]:
thymio_data = []

def get_data():
    thymio_data.append({"ground":list(node["prox.ground.reflected"]), 
                        "sensor":list(node["prox.ground.reflected"]),
                        "left_speed":node["motor.left.speed"],
                        "right_speed":node["motor.right.speed"]})

In [2]:
# t1 = time.time()
# for _ in range(n):
#     dt = time.time() - t1
#     # get_data()
#     kalman_pos()
#     t1 = time.time()

In [ ]:
L = 93
var_r, var_l = 0, 0
m = np.zeros((3, 3))
M = np.array([[var_r, 0], [0, var_l]])
R = np.cov(m)

In [ ]:
def wrap_angle(a):
    return (a + np.pi) % (2 * np.pi) - np.pi

In [ ]:
def f(x, y, theta, v_r, v_l, dt, L=L):
    f = np.array([x + (v_r + v_l) * np.cos(theta) * dt / 2, 
         y + (v_r + v_l) * np.sin(theta) * dt / 2, 
         wrap_angle(theta + (v_r - v_l) * dt / L)])
    return f

In [ ]:
def kalman(x_old, v_r, v_l, P_old, dt, M=M, R=R, x_measured=None):
    # Units : 
    # x, y, L in mm; theta in radian (-π, π)
    # v_r, v_l in mm/s
    # dt in s
    # M in (mm/s)^2
    # R and P in diag(mm^2, mm^2, rad^2)
    # All arrays should be NumPy arrays of floats
    x, y, theta = x_old[0], x_old[1], x_old[2]
    G = np.array([[np.cos(theta) * dt / 2, np.cos(theta) * dt / 2], 
                  [np.sin(theta) * dt / 2, np.sin(theta) * dt / 2], 
                  [dt / 2, -dt / 2]])
    F = np.array([[1, 0, -(v_r + v_l) * np.sin(theta) * dt / 2], 
         [0, 1, (v_r + v_l) * np.cos(theta) * dt / 2], 
         [0, 0, 1]])
    x_pred = f(x, y, theta, v_r, v_l, dt)
    P_pred = F @ P_old @ F.T + G @ M @ G.T
    if x_measured is not None:
        i = x_measured - x_pred
        i[2] = wrap_angle(i[2])
        S = P_pred + R
        K = P_pred @ np.linalg.inv(S)
        x_est = x_pred + K @ i
        x_est[2] = wrap_angle(x_est[2])
        P_est = (np.identity(3) - K) @ P_pred
        return x_est, P_est
    else : 
        return x_pred, P_pred